In [22]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, KFold, cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_squared_error, mean_absolute_error

from xgboost import XGBRegressor
from scipy.stats import randint

import os
import joblib

In [2]:
# 1- xgboost
# 2- adaboost
# 3- gradient boosting
# 4- catboost
# 5- lightboost
# 6- random forest
# 7- stacking 01
# 8- stacking 02

In [3]:
df = pd.read_csv("../data/preprocessed_dataset.csv")

df.head(2)

,FA (kg/m3),GGBFS (kg/m3),Coarse aggregate (kg/m3),Fine aggregate (kg/m3),Na2SiO3,NaOH,Sio2 (Dry),Concentration (M) NaOH,Superplasticizer,Total water,Initial curing time (day),Initial curing temp (C),Initial curing rest time (day),Final curing temp (C),Water_to_Binder,Water_to_Activator,SP_ratio,total_material_mass,Total_CO2_kg,Compressive_strength_MPa
0,0.000000,0.069565,-0.127973,-0.495765,0.700467,0.741549,0.756779,0.0,0.095238,0.651360,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.107509,0.885142,512.210033,46.57
1,0.140762,0.162319,-0.465568,0.198785,0.900600,0.950927,0.972882,0.0,0.142857,0.849623,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.159822,0.948367,534.887466,46.01


In [4]:
labels = df[['Total_CO2_kg', 'Compressive_strength_MPa']]
df.drop(columns=['Total_CO2_kg', 'Compressive_strength_MPa'], inplace=True)

In [5]:
df.shape

(276, 18)

In [18]:
X_train, X_test, y_train, y_test = train_test_split(df, labels, random_state=42, test_size=0.20)

In [7]:
len(X_train)

220

# xgboost


In [8]:
xgb_model = XGBRegressor(random_state=42)

param_dist = {
    'n_estimators': randint(50, 500),  
    'learning_rate': np.linspace(0.01, 0.3, 30),  
    'max_depth': randint(3, 15),  
    'min_child_weight': randint(1, 10), 
    'subsample': np.linspace(0.6, 1.0, 5), 
    'colsample_bytree': np.linspace(0.6, 1.0, 5),  
    'gamma': np.linspace(0, 1, 10) 
}

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=100,  
     scoring="r2",  
    cv=10,  
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best Hyperparameters:", random_search.best_params_)
print("Best R2:", random_search.best_score_)


Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Best Hyperparameters: {'colsample_bytree': 0.8, 'gamma': 0.8888888888888888, 'learning_rate': 0.03, 'max_depth': 11, 'min_child_weight': 2, 'n_estimators': 433, 'subsample': 0.7}
Best R2: 0.9674317538738251


In [10]:
best = random_search.best_params_

fixed_params = {
    "subsample": best["subsample"],
    "colsample_bytree": best["colsample_bytree"],
    "gamma": best["gamma"],
}


grid = {
    "n_estimators": [
        best["n_estimators"] - 60,
        best["n_estimators"] - 30,
        best["n_estimators"],
        best["n_estimators"] + 30,
        best["n_estimators"] + 60,
    ],

    "learning_rate": [
        best["learning_rate"] * 0.7,
        best["learning_rate"] * 0.85,
        best["learning_rate"],
        best["learning_rate"] * 1.15,
        best["learning_rate"] * 1.3,
    ],

    "max_depth": [
        max(2, best["max_depth"] - 2),
        max(2, best["max_depth"] - 1),
        best["max_depth"],
        best["max_depth"] + 1,
    ],

    "min_child_weight": [
        max(1, best["min_child_weight"] - 2),
        max(1, best["min_child_weight"] - 1),
        best["min_child_weight"],
        best["min_child_weight"] + 1,
    ],
}



xgb_local = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    **fixed_params
)

cv = KFold(n_splits=10, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_local,
    param_grid=grid,
    scoring="r2",
    cv=cv,
    verbose=2,
    n_jobs=4,         
    error_score="raise"
)

grid_search.fit(X_train, y_train)

print("Best params (Grid):", grid_search.best_params_)
print("Best R2 (Grid):", grid_search.best_score_)


Fitting 10 folds for each of 400 candidates, totalling 4000 fits
Best params (Grid): {'learning_rate': 0.0255, 'max_depth': 10, 'min_child_weight': 2, 'n_estimators': 493}
Best R2 (Grid): 0.9702460408210755


In [14]:
best_params = grid_search.best_params_

model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    **best_params
)

cv = KFold(n_splits=10, shuffle=True, random_state=42)

scoring = {
    "r2": "r2",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error"
}

cv_res = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=4,
    return_train_score=False
)

print("CV R2 mean ± std:", cv_res["test_r2"].mean(), "±", cv_res["test_r2"].std())
print("CV RMSE mean ± std:", (-cv_res["test_rmse"]).mean(), "±", (-cv_res["test_rmse"]).std())
print("CV MAE mean ± std:", (-cv_res["test_mae"]).mean(), "±", (-cv_res["test_mae"]).std())

CV R2 mean ± std: 0.9644635438919067 ± 0.013318186915879514
CV RMSE mean ± std: 4.61124439239502 ± 1.6960367641799936
CV MAE mean ± std: 2.7825350999832152 ± 0.8184673434739512


In [15]:
model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [20]:
y_pred = model.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
mae_test = mean_absolute_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)

print("TEST R2:", r2_test)
print("TEST RMSE:", rmse_test)
print("TEST MAE:", mae_test)

TEST R2: 0.9765200614929199
TEST RMSE: 3.6186440091596266
TEST MAE: 1.8707969188690186


In [23]:
model_dir = "../ml_models"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, "xgb_final_model.pkl")
joblib.dump(model, model_path)

print(f"Model saved to: {model_path}")

Model saved to: ../ml_models\xgb_final_model.pkl


In [26]:
a = model.predict(X_test)
a

array([[468.77002 ,  26.522903],
       [463.9101  ,  50.543896],
       [310.63858 ,  47.727898],
       [426.84464 ,  17.941238],
       [361.8265  ,  48.72156 ],
       [491.56845 ,  46.014603],
       [364.27612 ,  55.490005],
       [304.6252  ,  41.78771 ],
       [534.0538  ,  52.72731 ],
       [430.0856  ,  41.31065 ],
       [420.11264 ,  11.946862],
       [414.47714 ,  47.915543],
       [569.4415  ,  53.465   ],
       [442.82785 ,  27.951231],
       [422.71417 ,  26.112062],
       [298.1065  ,  39.77476 ],
       [412.27188 ,  62.34778 ],
       [475.77374 ,  49.755116],
       [518.4202  ,  24.06482 ],
       [441.59412 ,  49.16014 ],
       [461.66815 ,  55.20233 ],
       [483.93024 ,  47.016445],
       [479.56213 ,  47.284912],
       [397.81485 ,  34.838947],
       [442.4862  ,  41.244278],
       [458.40973 ,  56.831924],
       [481.44815 ,  57.184776],
       [548.00055 ,  47.08545 ],
       [367.76306 ,  50.775646],
       [540.5176  ,  21.848345],
       [30